# DTCR-U-Net: Kaggle Training & Evaluation Pipeline
This notebook executes the complete end-to-end training and evaluation of the **Dual-Task Consistency Regularization and Multi-Scale Adaptive Attention Network (DTCR-U-Net)** strictly aligned with the research paper.

---
### Cell 1: Ensure Kaggle GPU is Enabled & Setup Environment
Verifies GPU hardware acceleration (e.g. Tesla T4, P100), pulls the latest code, and installs required packages.

In [ ]:
# 1. Ensure Kaggle GPU is Enabled
import sys
import torch

print('=' * 60)
print('           KAGGLE ACCELERATOR & GPU VERIFICATION')
print('=' * 60)
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU Acceleration Active: {gpu_count}x {gpu_name}')
    print(f'✅ Dedicated VRAM Available: {vram_gb:.2f} GB')
    print(f'✅ PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
else:
    print('❌ GPU NOT DETECTED!')
    print('👉 Please enable GPU: Settings (right sidebar) -> Accelerator -> GPU T4 x2 or P100')
print('=' * 60)

# 2. Clone Repository & Install Dependencies
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git || (cd fyp-dtcr-unet && git pull origin main)
%cd fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt


---
### Cell 2: Discover Input Datasets
Scans `/kaggle/input/` and auto-detects attached benchmark datasets (MosMedData, LIDC-IDRI, COVID-19-CT-Seg).

In [ ]:
# Discover Input Datasets in /kaggle/input
import os
from pathlib import Path

print('=' * 60)
print('             DISCOVERING KAGGLE INPUT DATASETS')
print('=' * 60)

input_root = Path('/kaggle/input')
if not input_root.exists():
    print('❌ /kaggle/input directory not found.')
else:
    for d in sorted(input_root.iterdir()):
        if d.is_dir():
            n_files = sum(1 for _ in d.rglob('*') if _.is_file())
            print(f'📁 {d.name}/ ({n_files} files)')

# Auto-detect MosMedData (contains study_*.nii or study_*.nii.gz)
MOSMED_PATH = None
for d in input_root.iterdir():
    if list(d.rglob('study_*.nii*')):
        MOSMED_PATH = str(d)
        print(f'\n✅ MosMedData found at: {MOSMED_PATH}')
        break
if not MOSMED_PATH:
    print('\n⚠️ MosMedData NOT found in /kaggle/input/')

# Auto-detect LIDC-IDRI (contains LIDC-IDRI-* or seg_LIDC-IDRI)
LIDC_PATH = None
for d in input_root.iterdir():
    if list(d.rglob('LIDC-IDRI-*')) or list(d.rglob('seg_LIDC-IDRI')):
        LIDC_PATH = str(d)
        print(f'✅ LIDC-IDRI found at: {LIDC_PATH}')
        break
if not LIDC_PATH:
    print('⚠️ LIDC-IDRI NOT found in /kaggle/input/')

# Auto-detect COVID-19-CT-Seg
COVID_PATH = None
for d in input_root.iterdir():
    if list(d.rglob('*covid19_ct*')) or list(d.rglob('*COVID*CT*')):
        COVID_PATH = str(d)
        print(f'✅ COVID-19-CT-Seg found at: {COVID_PATH}')
        break

PROCESSED_OUT = '/kaggle/working/data/processed'
print(f'\nTarget Preprocessed Directory: {PROCESSED_OUT}')
print('=' * 60)


---
### Cell 3: Run Data Preprocessing
Preprocesses CT volumes according to paper Section 4.1: in-plane resampling, $[-1000, 400]$ HU windowing, CLAHE normalization, $256 \times 256$ spatial sizing, context window of 2 adjacent slices above and below ($k=2$, 5 slices total), and creates the 40% labeled / 60% unlabeled semi-supervised patient split.

In [ ]:
# Run Data Preprocessing (Paper Specs: 256x256, k=2 Context Slices, 40% Labeled)
import os
from pathlib import Path

os.makedirs(PROCESSED_OUT, exist_ok=True)

# Build command dynamically based on discovered datasets
cmd = (
    f'python data/preprocessing.py '
    f'--out {PROCESSED_OUT} '
    f'--size 256 '
    f'--context 2 '
    f'--labeled-ratio 0.4 '
    f'--max-patients 150'
)

if MOSMED_PATH:
    cmd += f' --mosmed "{MOSMED_PATH}"'
if LIDC_PATH:
    cmd += f' --lidc "{LIDC_PATH}"'
if COVID_PATH:
    cmd += f' --covid19 "{COVID_PATH}"'

print(f'Executing Preprocessing:\n{cmd}\n')
!{cmd}

# Verify extracted slice counts across splits
train_slices = len(list(Path(PROCESSED_OUT).glob('train/images/*.npy')))
val_slices = len(list(Path(PROCESSED_OUT).glob('val/images/*.npy')))
test_slices = len(list(Path(PROCESSED_OUT).glob('test/images/*.npy')))
print(f'\n✅ Preprocessing Complete! Slices: {train_slices} Train | {val_slices} Val | {test_slices} Test')


---
### Cell 4: Train DTCR-U-Net Model
Trains the 4-stage DTCR-U-Net (18.65M parameter budget) using AdamW (lr=0.001, weight decay=1e-4), Cosine Annealing, AMP mixed precision, dynamic consistency weight $\lambda_d(t)$, $\lambda_1=0.3$, $\lambda_2=1.0$, and 80 epochs.

In [ ]:
# Train DTCR-U-Net (Paper Benchmark Protocol: 4-Stage, 80 Epochs, Dual-Task Consistency)
!python train.py \
    --data {PROCESSED_OUT} \
    --num-stages 4 \
    --epochs 80 \
    --batch-size 8 \
    --grad-accum 4 \
    --lr 0.001 \
    --weight-decay 1e-4 \
    --labeled-ratio 0.4 \
    --beta 0.5 \
    --lambda1 0.3 \
    --lambda2 1.0 \
    --lambda-context 0.0 \
    --consistency-weight 0.1 \
    --dice-weight 1.0 \
    --bce-weight 0.0 \
    --val-threshold 0.50 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs


---
### Cell 5: Evaluate Best Model on Test Dataset
Loads `/kaggle/working/checkpoints/best_model.pth` and evaluates on the withheld test dataset, computing patient-level 3D volume metrics (Dice, Sensitivity, Specificity, Accuracy, F1) matching Paper Table 2.

In [ ]:
# Evaluate Best Model on Test Dataset (Computes 3D Volume & 2D Metrics)
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data {PROCESSED_OUT} \
    --batch-size 8 \
    --threshold 0.50 \
    --save-vis 16 \
    --save-predictions \
    --out /kaggle/working/evaluation_results


---
### Cell 6: Visual Prediction Comparison
Renders side-by-side qualitative comparisons (Raw CT Slice | Ground Truth Mask | DTCR-U-Net Prediction | Level-Set LSF Map) directly in the notebook.

In [ ]:
# Visual Prediction Comparison (Raw CT | Ground Truth | DTCR-U-Net Pred | Level Set LSF)
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

comparison_images = sorted(glob.glob('/kaggle/working/evaluation_results/comparisons/*.png'))

if not comparison_images:
    print('❌ No visual comparison images found. Check /kaggle/working/evaluation_results/comparisons/')
else:
    print(f'✅ Found {len(comparison_images)} visual prediction comparisons. Displaying top 8:')
    show_images = comparison_images[:8]
    
    fig, axes = plt.subplots(len(show_images), 1, figsize=(18, 4.5 * len(show_images)))
    if len(show_images) == 1:
        axes = [axes]
        
    for i, img_path in enumerate(show_images):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f'Visual Prediction #{i+1}: {Path(img_path).stem}', fontsize=13, pad=8)
        
    plt.tight_layout()
    plt.show()
